# 02_2 DQA Functionality Probe 7h

このNotebookは、DQAがどういう条件なら機能するのかを広めに検証するための実験ノートです。

見るもの:

- server側が本当にcloudy/partly cloudyなのか、client側がovercast/rainy/snowyなのかのデータ集計
- warmupをそのまま評価したbaseline
- cloudy server GTだけで追加学習した上限寄りbaseline
- client pseudo-label学習の条件違い: 閾値、LR、train scope
- client local checkpointがserver valでどれくらい壊れるか
- FedAvg vs DQA v2 default vs conservative DQA vs tiny-residual DQA
- aggregation直後とserver calibration後のmAP比較

出力はすべて `dynamic_quality_aware_classwise_aggregation/exploring/runs/02_2_dqa_functionality_probe_7h/` に閉じます。既存のDQA/FedSTO本番実験は触りません。

## 1. Setup

デフォルトでは `dqa_v2_scene_12h` のwarmup checkpointを使います。7時間枠に収まるよう、server train/valとclient targetはmini listを使います。途中で止まっても、既存checkpointやJSONがあればskipしてresumeします。

In [1]:
print("Hello, World!")

Hello, World!


In [2]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name != "Object_Detection" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "Object_Detection":
    raise RuntimeError("Run this notebook from inside /app/Object_Detection")

EXPLORING_ROOT = ROOT / "dynamic_quality_aware_classwise_aggregation" / "exploring"
if str(EXPLORING_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPLORING_ROOT))

from dqa_probe_02_2 import ProbeConfig, DQAFunctionalityProbe

cfg = ProbeConfig(
    source_warmup_key="dqa_v2_scene_12h",
    experiment_name="02_2_dqa_functionality_probe_7h",
    server_train_limit=2048,
    server_val_limit=1024,
    client_target_limit=2048,
    max_wall_hours=7.0,
    device_cli="0",
    batch_size=8,
    val_batch_size=1,
    workers=0,
    force_rerun=False,
    run_client_local_eval=True,
    run_server_calibration=True,
)

probe = DQAFunctionalityProbe(cfg)
probe.describe()

{'experiment_root': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/exploring/runs/02_2_dqa_functionality_probe_7h',
 'warmup': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/global_checkpoints/round000_warmup.pt',
 'client_variants': ['ssod_default_all_lr1e-2',
  'ssod_default_all_lr1e-3',
  'ssod_strict_all_lr3e-4',
  'ssod_strict_neck_head_lr1e-3',
  'ssod_very_strict_all_lr3e-4'],
 'aggregation_variants': ['fedavg',
  'dqa_v2_default',
  'dqa_v2_conservative',
  'dqa_v2_tiny_residual'],
 'calibration_variants': ['cal_all_lr3e-4_ep1', 'cal_all_lr1e-3_ep1'],
 'budget_hours': 7.0}

## 2. Data Audit

まず、serverがcloudy相当のpartly cloudyで、clientがovercast/rainy/snowy targetになっていること、さらにserver GTのclass分布を確認します。client targetはunlabeledなので画像数だけを集計します。

In [3]:
data_audit = probe.prepare_data()
cols = ["role", "weather", "has_gt", "images", "objects", "missing_labels", "empty_labels"]
cols = [c for c in cols if c in data_audit.columns]
data_audit[cols]

,role,weather,has_gt,images,objects,missing_labels,empty_labels
0,server_train,partly_cloudy,True,4881,97123.0,0.0,0.0
1,server_val,partly_cloudy,True,738,14937.0,0.0,0.0
2,mini_server_train,partly_cloudy,True,2048,41434.0,0.0,0.0
3,mini_server_val,partly_cloudy,True,738,14937.0,0.0,0.0
4,client_0_target_full,overcast,False,5000,NaN,NaN,NaN
5,client_0_target_mini,overcast,False,2048,NaN,NaN,NaN
6,client_1_target_full,rainy,False,5000,NaN,NaN,NaN
7,client_1_target_mini,rainy,False,2048,NaN,NaN,NaN
8,client_2_target_full,snowy,False,5000,NaN,NaN,NaN
9,client_2_target_mini,snowy,False,2048,NaN,NaN,NaN


In [4]:
gt_cols = [c for c in data_audit.columns if c.startswith("gt_")]
server_gt = data_audit[data_audit["has_gt"].eq(True)][["role", "images", "objects"] + gt_cols]
server_gt

,role,images,objects,gt_person,gt_rider,gt_car,gt_bus,gt_truck,gt_bike,gt_motor,gt_traffic light,gt_traffic sign,gt_train
0,server_train,4881,97123.0,6370.0,376.0,55752.0,1112.0,2993.0,488.0,270.0,11415.0,18325.0,22.0
1,server_val,738,14937.0,1052.0,44.0,8622.0,151.0,467.0,70.0,65.0,1619.0,2845.0,2.0
2,mini_server_train,2048,41434.0,2740.0,169.0,23738.0,482.0,1245.0,198.0,135.0,5010.0,7702.0,15.0
3,mini_server_val,738,14937.0,1052.0,44.0,8622.0,151.0,467.0,70.0,65.0,1619.0,2845.0,2.0


## 3. Run The Full Probe

このセルが本体です。内部では次を順に実行します。

1. warmup baseline evaluation
2. cloudy server supervised continuation baseline
3. 5種類のclient pseudo-label学習 x 3 clients
4. client local checkpoint evaluation
5. 4種類のaggregation比較
6. aggregation後のserver calibration比較
7. leaderboard作成

`force_rerun=False` なので、途中まで終わっていれば再実行時にskipします。

In [ ]:
leaderboard = probe.run_all()
display_cols = [
    "name",
    "kind",
    "variant",
    "aggregation",
    "calibration",
    "status",
    "map50_final",
    "map50_95_final",
    "delta_map50_95_vs_warmup",
    "precision_final",
    "recall_final",
    "source_table",
]
display_cols = [c for c in display_cols if c in leaderboard.columns]
leaderboard[display_cols].head(40)

## 4. Pseudo-Label Statistics

DQAが効くかどうかは、client pseudo labelの分布にかなり依存します。ここではclass偏り、active class数、top class shareを確認します。`top_class_share` が高すぎる場合、DQAがconfidenceをqualityとして見ても実際には偏った更新になりやすいです。

In [ ]:
pseudo = probe.summarize_pseudo_stats()
pseudo_cols = [
    "variant",
    "client_id",
    "weather",
    "status",
    "pseudo_total",
    "active_classes",
    "zero_classes",
    "top_class_share",
    "mean_quality_active",
]
pseudo[pseudo_cols].sort_values(["variant", "client_id"])

In [ ]:
pseudo_class_cols = [c for c in pseudo.columns if c.startswith("pseudo_")]
pseudo[["variant", "client_id", "weather"] + pseudo_class_cols].sort_values(["variant", "client_id"])

## 5. Aggregation And Calibration Tables

DQAが機能しているなら、少なくとも同じclient checkpointsからのFedAvgよりDQA系aggregationが良くなるはずです。逆にDQAでも落ちるなら、client pseudo update自体が強すぎるか、pseudo statsがqualityとして信用できていない可能性が高いです。

In [ ]:
import pandas as pd

result_root = probe.result_root
agg = pd.read_csv(result_root / "aggregation_summary.csv") if (result_root / "aggregation_summary.csv").exists() else pd.DataFrame()
cal = pd.read_csv(result_root / "server_calibration_summary.csv") if (result_root / "server_calibration_summary.csv").exists() else pd.DataFrame()

agg_cols = ["variant", "aggregation", "status", "map50_final", "map50_95_final", "precision_final", "recall_final", "active_classes", "aggregation_note"]
agg_cols = [c for c in agg_cols if c in agg.columns]
agg[agg_cols].sort_values("map50_95_final", ascending=False, na_position="last").head(30)

In [ ]:
cal_cols = ["variant", "aggregation", "calibration", "status", "map50_final", "map50_95_final", "precision_final", "recall_final", "note"]
cal_cols = [c for c in cal_cols if c in cal.columns]
cal[cal_cols].sort_values("map50_95_final", ascending=False, na_position="last").head(30)

## 6. Decision Rules

読み方の目安:

- `server_oracle_cloudy_all_lr3e-4_ep3` が強い: warmup後に伸びる余地はあるので、問題は主にclient pseudo updateかaggregation。
- `client_local_eval` がwarmupより大きく落ちる: client pseudo-label学習がモデルを壊している。
- FedAvgよりDQA v2が良い: quality-aware aggregationは一定機能している。
- DQA defaultよりconservative/tiny residualが良い: DQAは効くがclient residualをもっと小さくする必要がある。
- strict pseudo + low LRだけ良い: 今のphase2 pseudo-label閾値/LRが強すぎる。
- aggregation直後は悪いがcalibration後に戻る: server updateをDQA後の必須repairとして設計すべき。
- calibration後でもwarmup未満: client更新を採用しないguard/rollbackが必要。

最終的には `overall_leaderboard.csv`, `pseudo_stats_summary.csv`, `aggregation_summary.csv`, `server_calibration_summary.csv` を見て、DQAが機能する条件を決めます。